# Ψ-NeuroFractal : Detection precoce Alzheimer/Parkinson
## Entrainement du modele ΨTA vs CNN Baseline

In [ ]:
!git clone https://github.com/hassanegbane190-max/psi-neurofractal.git
%cd psi-neurofractal
!pip install torch numpy scipy scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from scipy.signal import hilbert

np.random.seed(42)
torch.manual_seed(42)

N_CHANNELS = 64
N_TIMES = 256

# ============================================================
# NEURONE ΨTA
# ============================================================
class AsymmetricPsiTNeuron(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weights = nn.Parameter(torch.randn(out_features, in_features, dtype=torch.complex64) * 0.8)
        self.bias = nn.Parameter(torch.zeros(out_features, dtype=torch.complex64))
        self.alpha = nn.Parameter(torch.ones(out_features) * 1.5)
        self.beta = nn.Parameter(torch.zeros(out_features))
        self.gamma = nn.Parameter(torch.ones(out_features) * 2.0)

    def forward(self, x):
        z = torch.matmul(x, self.weights.t()) + self.bias
        R = torch.abs(z)
        theta = torch.angle(z)
        amplitude = torch.tanh(R)
        angle_asym = torch.relu(theta + self.beta)
        phi_att = np.pi * self.gamma * torch.tanh(self.alpha * R) * torch.sin(angle_asym)
        return amplitude * torch.complex(torch.cos(phi_att), torch.sin(phi_att))

# ============================================================
# DATASET EEG SIMULE
# ============================================================
class EEGAlzheimerDataset(Dataset):
    def __init__(self, n_channels=64, n_times=256, n_per_class=500):
        self.samples = []
        self.labels = []
        fs = 256.0
        t = np.arange(n_times) / fs
        for cls in range(2):
            for _ in range(n_per_class):
                eeg = np.zeros((n_channels, n_times))
                for ch in range(n_channels):
                    if cls == 0:
                        a = 10*np.sin(2*np.pi*10*t + np.random.uniform(0,2*np.pi))
                        b = 3*np.sin(2*np.pi*20*t + np.random.uniform(0,2*np.pi))
                        th = 4*np.sin(2*np.pi*6*t + np.random.uniform(0,2*np.pi))
                        eeg[ch] = a + b + th + 0.5*np.random.randn(len(t))
                    else:
                        a = 2*np.sin(2*np.pi*10*t + np.random.uniform(0,2*np.pi))
                        th = 8*np.sin(2*np.pi*5*t + np.random.uniform(0,2*np.pi))
                        d = 6*np.sin(2*np.pi*2*t + np.random.uniform(0,2*np.pi))
                        chaos = sum((1.0/k)*np.sin(2*np.pi*k*3.7*t + np.random.uniform(0,2*np.pi)) for k in range(1,6))
                        eeg[ch] = a + th + d + chaos + 1.5*np.random.randn(len(t))
                cx = np.zeros((n_channels, n_times), dtype=np.complex64)
                for ch in range(n_channels):
                    cx[ch] = hilbert(eeg[ch]).astype(np.complex64)
                self.samples.append(torch.tensor(cx, dtype=torch.complex64))
                self.labels.append(cls)
        self.samples = torch.stack(self.samples)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.samples[i], self.labels[i]

# ============================================================
# MODELES
# ============================================================
class NeuroFractalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.channel_proj = nn.Linear(N_TIMES, 1, dtype=torch.complex64)
        self.psi1 = AsymmetricPsiTNeuron(N_CHANNELS, 128)
        self.psi2 = AsymmetricPsiTNeuron(128, 64)
        self.norm1 = nn.LayerNorm([256])
        self.norm2 = nn.LayerNorm([128])
        self.classifier = nn.Sequential(
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        x = self.channel_proj(x).squeeze(-1)
        h = self.psi1(x)
        hr = torch.cat([torch.real(h), torch.imag(h)], dim=1)
        hr = self.norm1(hr)
        h = self.psi2(h)
        hr = torch.cat([torch.real(h), torch.imag(h)], dim=1)
        hr = self.norm2(hr)
        return self.classifier(hr)

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(N_TIMES, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Sequential(nn.Linear(N_CHANNELS*64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 2))

    def forward(self, x):
        if x.dim() == 3: x = x.squeeze(1)
        if x.is_complex(): x = torch.cat([torch.real(x), torch.imag(x)], dim=1)
        x = x.float()
        feats = [self.features(x[:, ch, :]) for ch in range(x.shape[1])]
        return self.classifier(torch.cat(feats, dim=1))

print('Definitions chargees !')

In [ ]:
print('Generation des donnees EEG...')
dataset = EEGAlzheimerDataset()
print(f'Dataset : {len(dataset)} echantillons | Shape : {dataset.samples.shape}')
print(f'Labels : Sain={int((dataset.labels==0).sum())} | Pathologique={int((dataset.labels==1).sum())}')

n_train = 800
train_loader = DataLoader(Subset(dataset, range(n_train)), batch_size=32, shuffle=True)
test_loader = DataLoader(Subset(dataset, range(n_train, 1000)), batch_size=32, shuffle=False)
print(f'Train : {n_train} | Test : 200')

## Entrainement ΨTA

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

model_psita = NeuroFractalNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_psita.parameters(), lr=0.001, weight_decay=1e-5)

print(f'Parametres ΨTA : {sum(p.numel() for p in model_psita.parameters()):,}')
print('\n' + '='*65)
print(f'{"Epoch":>6} | {"Loss Train":>10} | {"Acc Train":>10} | {"Loss Test":>10} | {"Acc Test":>10}')
print('='*65)

hist_p = {'tl':[], 'ta':[], 'vl':[], 'va':[]}
best_acc_p = 0
t0 = time.time()

for ep in range(1, 101):
    model_psita.train()
    tl = cor = tot = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_psita(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        tl += loss.item()*bx.size(0)
        cor += (out.argmax(1)==by).sum().item()
        tot += by.size(0)
    trl, tra = tl/tot, cor/tot*100

    model_psita.eval()
    tl = cor = tot = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            out = model_psita(bx)
            loss = criterion(out, by)
            tl += loss.item()*bx.size(0)
            cor += (out.argmax(1)==by).sum().item()
            tot += by.size(0)
    vdl, vda = tl/tot, cor/tot*100

    hist_p['tl'].append(trl); hist_p['ta'].append(tra)
    hist_p['vl'].append(vdl); hist_p['va'].append(vda)

    if vda > best_acc_p:
        best_acc_p = vda
        torch.save(model_psita.state_dict(), 'psita_best.pt')

    if ep % 10 == 0 or ep == 1:
        print(f'{ep:6d} | {trl:10.4f} | {tra:9.2f}% | {vdl:10.4f} | {vda:9.2f}%')

print('='*65)
print(f'ΨTA termine en {time.time()-t0:.1f}s | Meilleure : {best_acc_p:.2f}%')

## Entrainement CNN Baseline

In [ ]:
model_cnn = BaselineCNN().to(device)
opt_cnn = torch.optim.Adam(model_cnn.parameters(), lr=0.001, weight_decay=1e-5)

print(f'Parametres CNN : {sum(p.numel() for p in model_cnn.parameters()):,}')
print('\n' + '='*65)
print(f'{"Epoch":>6} | {"Loss Train":>10} | {"Acc Train":>10} | {"Loss Test":>10} | {"Acc Test":>10}')
print('='*65)

hist_c = {'tl':[], 'ta':[], 'vl':[], 'va':[]}
best_acc_c = 0
t0 = time.time()

for ep in range(1, 101):
    model_cnn.train()
    tl = cor = tot = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        opt_cnn.zero_grad()
        out = model_cnn(bx)
        loss = criterion(out, by)
        loss.backward()
        opt_cnn.step()
        tl += loss.item()*bx.size(0)
        cor += (out.argmax(1)==by).sum().item()
        tot += by.size(0)
    trl, tra = tl/tot, cor/tot*100

    model_cnn.eval()
    tl = cor = tot = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            out = model_cnn(bx)
            loss = criterion(out, by)
            tl += loss.item()*bx.size(0)
            cor += (out.argmax(1)==by).sum().item()
            tot += by.size(0)
    vdl, vda = tl/tot, cor/tot*100

    hist_c['tl'].append(trl); hist_c['ta'].append(tra)
    hist_c['vl'].append(vdl); hist_c['va'].append(vda)

    if vda > best_acc_c:
        best_acc_c = vda

    if ep % 10 == 0 or ep == 1:
        print(f'{ep:6d} | {trl:10.4f} | {tra:9.2f}% | {vdl:10.4f} | {vda:9.2f}%')

print('='*65)
print(f'CNN termine en {time.time()-t0:.1f}s | Meilleure : {best_acc_c:.2f}%')

## Resultats Finaux

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(hist_p['tl'], label='Train'); axes[0,0].plot(hist_p['vl'], label='Test'); axes[0,0].set_title('Loss - ΨTA', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(hist_p['ta'], label='Train'); axes[0,1].plot(hist_p['va'], label='Test'); axes[0,1].set_title('Accuracy - ΨTA', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(hist_c['tl'], label='Train'); axes[1,0].plot(hist_c['vl'], label='Test'); axes[1,0].set_title('Loss - CNN', fontweight='bold'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(hist_c['ta'], label='Train'); axes[1,1].plot(hist_c['va'], label='Test'); axes[1,1].set_title('Accuracy - CNN', fontweight='bold'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('comparaison.png', dpi=150)
plt.show()

print('\n' + '='*65)
print('  RESULTATS FINAUX')
print('='*65)
print(f'  ΨTA (NeuroFractal)  : {best_acc_p:.2f}%')
print(f'  CNN Baseline        : {best_acc_c:.2f}%')
print(f'  Gain ΨTA            : +{best_acc_p-best_acc_c:.2f}%')
print('='*65)